# 52. Browser Agent DOM stale action：怎样在点击前重新观察并验证目标？

## 面试回答主线

Browser Agent 的计划往往基于某一时刻的 DOM snapshot，但 React 重渲染、导航和异步列表更新会让 node_id 失效或被其他元素复用。最危险的实现是先定位、思考数秒后直接点击旧 node_id，因为它可能执行不可逆的删除、支付或授权动作。可靠流程是把意图绑定到稳定语义键，在动作前重新抓取 snapshot，重新解析 locator，并验证 role、name、enabled 与金额等前置条件。面试中我会输出 snapshot 指纹、旧节点实际指向、重新定位轨迹和 action receipt，而不是只写一个“元素存在”的断言。若前置条件发生变化，正确行为是停止并重新规划或请求确认，不是自动寻找相似按钮。生产环境还需要处理验证与点击之间的竞态、iframe、shadow DOM、导航提交以及审计截图。

## 1. 真实案例：六个页面在 Agent 思考期间发生重渲染

案例覆盖提交订单、删除草稿、预订会议室、禁用用户、更新购物车和接受隐私条款。每个页面的目标元素在新 snapshot 中获得新 node_id，而旧 node_id 被无关按钮复用；稳定语义键仍指向真正业务目标。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示 DOM 与动作意图
import hashlib  # 导入哈希函数生成可比较的 snapshot 指纹
import json  # 导入 JSON 序列化器固定 DOM 指纹输入
def make_case(case_id, intent, stable_key, role, name, new_node_id, stale_name):  # 构造一次具有真实语义的 DOM 重渲染案例
    before = [{"node_id": "n1", "stable_key": stable_key, "role": role, "name": name, "enabled": True}, {"node_id": "n2", "stable_key": f"help-{case_id}", "role": "link", "name": "帮助", "enabled": True}]  # 定义规划时目标和帮助链接
    after = [{"node_id": "n1", "stable_key": f"stale-{case_id}", "role": "button", "name": stale_name, "enabled": True}, {"node_id": new_node_id, "stable_key": stable_key, "role": role, "name": name, "enabled": True}]  # 模拟旧 ID 被其他按钮复用且目标获得新 ID
    plan = {"intent": intent, "stable_key": stable_key, "role": role, "name": name}  # 保存与 node_id 解耦的动作计划和前置条件
    return {"id": case_id, "before": before, "after": after, "plan": plan}  # 返回完整页面演化案例
cases = [make_case("B01", "提交金额 699 元的订单", "checkout-submit", "button", "提交订单", "n9", "返回购物车"), make_case("B02", "删除标题为周报的草稿", "draft-delete-weekly", "button", "删除草稿", "n7", "删除全部"), make_case("B03", "预订 B201 会议室", "room-book-b201", "button", "预订 B201", "n8", "取消预订"), make_case("B04", "禁用测试用户 U17", "user-disable-u17", "button", "禁用 U17", "n6", "禁用管理员"), make_case("B05", "把商品 P3 数量更新为 2", "cart-update-p3", "button", "更新数量", "n5", "清空购物车"), make_case("B06", "接受当前隐私条款", "privacy-accept-v4", "button", "接受条款", "n4", "拒绝全部")]  # 定义六个不可随意点错的真实浏览器动作
preview = [{"案例": item["id"], "意图": item["plan"]["intent"], "旧node": item["before"][0]["node_id"], "重渲染后目标node": item["after"][1]["node_id"], "旧node现在指向": item["after"][0]["name"]} for item in cases]  # 汇总 stale action 的关键输入字段
print("Browser Agent DOM 演化案例预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示六个旧 node_id 被复用的危险页面

Browser Agent DOM 演化案例预览：
[{'案例': 'B01',
  '意图': '提交金额 699 元的订单',
  '旧node': 'n1',
  '重渲染后目标node': 'n9',
  '旧node现在指向': '返回购物车'},
 {'案例': 'B02',
  '意图': '删除标题为周报的草稿',
  '旧node': 'n1',
  '重渲染后目标node': 'n7',
  '旧node现在指向': '删除全部'},
 {'案例': 'B03',
  '意图': '预订 B201 会议室',
  '旧node': 'n1',
  '重渲染后目标node': 'n8',
  '旧node现在指向': '取消预订'},
 {'案例': 'B04',
  '意图': '禁用测试用户 U17',
  '旧node': 'n1',
  '重渲染后目标node': 'n6',
  '旧node现在指向': '禁用管理员'},
 {'案例': 'B05',
  '意图': '把商品 P3 数量更新为 2',
  '旧node': 'n1',
  '重渲染后目标node': 'n5',
  '旧node现在指向': '清空购物车'},
 {'案例': 'B06',
  '意图': '接受当前隐私条款',
  '旧node': 'n1',
  '重渲染后目标node': 'n4',
  '旧node现在指向': '拒绝全部'}]


## 2. Baseline（基线）：规划时缓存 node_id，执行时直接点击

错误基线在 before snapshot 中找到目标并缓存 `n1`，思考结束后在 after snapshot 仍按 `n1` 点击。由于 ID 被复用，每条案例都会击中语义完全不同的按钮；DOM 查询本身没有报错，所以这种事故比“元素不存在”更难发现。

In [2]:
def find_by_node_id(nodes, node_id):  # 在当前 DOM snapshot 中按瞬时 node_id 查找元素
    return next((node for node in nodes if node["node_id"] == node_id), None)  # 返回相同 ID 的当前元素或空值
baseline_rows = []  # 收集旧 node_id 直接执行的逐案例结果
for item in cases:  # 遍历六个真实浏览器任务
    planned_node = next(node for node in item["before"] if node["stable_key"] == item["plan"]["stable_key"])  # 在规划时 snapshot 中找到正确目标
    cached_node_id = planned_node["node_id"]  # 错误地只保存易失的 DOM node_id
    clicked_node = find_by_node_id(item["after"], cached_node_id)  # 重渲染后直接按旧 ID 执行动作
    hit_target = clicked_node is not None and clicked_node["stable_key"] == item["plan"]["stable_key"]  # 检查实际点击语义是否仍等于原意图
    baseline_rows.append({"案例": item["id"], "意图": item["plan"]["intent"], "缓存node": cached_node_id, "实际点击": clicked_node["name"] if clicked_node else "元素已消失", "命中目标": hit_target})  # 保存可观察的错误动作
print("旧 node_id 直接点击的基线结果：")  # 标注当前输出属于 stale action 基线
pprint(baseline_rows, sort_dicts=False)  # 展示每个危险动作实际点到了什么元素

旧 node_id 直接点击的基线结果：
[{'案例': 'B01',
  '意图': '提交金额 699 元的订单',
  '缓存node': 'n1',
  '实际点击': '返回购物车',
  '命中目标': False},
 {'案例': 'B02',
  '意图': '删除标题为周报的草稿',
  '缓存node': 'n1',
  '实际点击': '删除全部',
  '命中目标': False},
 {'案例': 'B03',
  '意图': '预订 B201 会议室',
  '缓存node': 'n1',
  '实际点击': '取消预订',
  '命中目标': False},
 {'案例': 'B04',
  '意图': '禁用测试用户 U17',
  '缓存node': 'n1',
  '实际点击': '禁用管理员',
  '命中目标': False},
 {'案例': 'B05',
  '意图': '把商品 P3 数量更新为 2',
  '缓存node': 'n1',
  '实际点击': '清空购物车',
  '命中目标': False},
 {'案例': 'B06', '意图': '接受当前隐私条款', '缓存node': 'n1', '实际点击': '拒绝全部', '命中目标': False}]


## 3. 手写核心机制：snapshot 指纹、语义重定位与前置条件

动作前重新观察页面并生成指纹，然后用 stable_key 重定位。找到元素后还必须逐项验证 role、可访问名称和 enabled；stable_key 只是候选定位，不是自动授权。验证通过后生成 receipt，记录动作绑定的 snapshot 与最终 node_id。

In [3]:
def snapshot_fingerprint(nodes):  # 为一次 DOM 观察生成稳定内容指纹
    serialized = json.dumps(nodes, ensure_ascii=False, sort_keys=True)  # 用固定字段顺序序列化当前 DOM 节点
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()[:12]  # 截取哈希作为可审计 snapshot 版本
def resolve_and_verify(nodes, plan):  # 在最新 snapshot 上重新解析目标并检查动作前置条件
    snapshot_id = snapshot_fingerprint(nodes)  # 记录本次动作依据的 DOM 内容版本
    candidates = [node for node in nodes if node["stable_key"] == plan["stable_key"]]  # 按稳定业务语义键重新查找目标
    if len(candidates) != 1:  # 要求目标在最新页面中唯一可解析
        raise ValueError(f"target_count={len(candidates)}")  # 对消失或重复元素显式停止
    node = candidates[0]  # 取得唯一候选后再检查安全属性
    observed = {"role": node["role"], "name": node["name"], "enabled": node["enabled"]}  # 提取会影响动作授权的当前属性
    expected = {"role": plan["role"], "name": plan["name"], "enabled": True}  # 构造规划时承诺的前置条件
    if observed != expected:  # 检查按钮语义、名称和可用状态是否全部未变
        raise ValueError(f"precondition_changed:{observed}")  # 拒绝在条件漂移后猜测或强点
    receipt = {"snapshot": snapshot_id, "node_id": node["node_id"], "stable_key": node["stable_key"], "action": "click", "verified": observed}  # 生成动作前验证回执
    return node, receipt  # 返回最新目标和可追溯 receipt
first_before_hash = snapshot_fingerprint(cases[0]["before"])  # 计算订单页面规划时 DOM 指纹
first_after_hash = snapshot_fingerprint(cases[0]["after"])  # 计算订单页面重渲染后的 DOM 指纹
resolved_node, first_receipt = resolve_and_verify(cases[0]["after"], cases[0]["plan"])  # 在最新订单页面重新定位并校验目标
print({"规划snapshot": first_before_hash, "执行snapshot": first_after_hash, "页面已变化": first_before_hash != first_after_hash, "动作receipt": first_receipt})  # 展示版本变化、重定位和验证证据

{'规划snapshot': 'c50e5306a9ae', '执行snapshot': 'ac1d0b4520c5', '页面已变化': True, '动作receipt': {'snapshot': 'ac1d0b4520c5', 'node_id': 'n9', 'stable_key': 'checkout-submit', 'action': 'click', 'verified': {'role': 'button', 'name': '提交订单', 'enabled': True}}}


## 4. 重新观察后执行：输出完整动作轨迹

下面对六个案例执行 observe → resolve → verify → click 状态机。点击结果以 stable_key 记账，避免只凭“click 没抛异常”判定成功。

In [4]:
safe_rows = []  # 收集六个页面的安全动作轨迹
receipts = {}  # 保存每个任务绑定最新 snapshot 的执行回执
for item in cases:  # 遍历全部真实浏览器动作
    latest_nodes = item["after"]  # 模拟动作前从浏览器重新抓取 DOM snapshot
    node, receipt = resolve_and_verify(latest_nodes, item["plan"])  # 重新定位并验证最新目标属性
    effect = f'clicked:{node["stable_key"]}'  # 用目标稳定语义键记录实际执行效果
    hit_target = node["stable_key"] == item["plan"]["stable_key"]  # 检查执行效果是否与原业务意图一致
    receipts[item["id"]] = receipt  # 保存当前动作的审计回执
    safe_rows.append({"案例": item["id"], "执行node": node["node_id"], "执行元素": node["name"], "snapshot": receipt["snapshot"], "效果": effect, "命中目标": hit_target})  # 保存逐案例执行轨迹
print("重新观察与验证后的动作轨迹：")  # 输出核心状态机结果标题
pprint(safe_rows, sort_dicts=False)  # 展示每个任务如何找到新的正确 node_id

重新观察与验证后的动作轨迹：
[{'案例': 'B01',
  '执行node': 'n9',
  '执行元素': '提交订单',
  'snapshot': 'ac1d0b4520c5',
  '效果': 'clicked:checkout-submit',
  '命中目标': True},
 {'案例': 'B02',
  '执行node': 'n7',
  '执行元素': '删除草稿',
  'snapshot': 'c3561586ed09',
  '效果': 'clicked:draft-delete-weekly',
  '命中目标': True},
 {'案例': 'B03',
  '执行node': 'n8',
  '执行元素': '预订 B201',
  'snapshot': 'cc32254e1deb',
  '效果': 'clicked:room-book-b201',
  '命中目标': True},
 {'案例': 'B04',
  '执行node': 'n6',
  '执行元素': '禁用 U17',
  'snapshot': 'ecc7100d58b3',
  '效果': 'clicked:user-disable-u17',
  '命中目标': True},
 {'案例': 'B05',
  '执行node': 'n5',
  '执行元素': '更新数量',
  'snapshot': '26f575a8268d',
  '效果': 'clicked:cart-update-p3',
  '命中目标': True},
 {'案例': 'B06',
  '执行node': 'n4',
  '执行元素': '接受条款',
  'snapshot': '0871e520a7ba',
  '效果': 'clicked:privacy-accept-v4',
  '命中目标': True}]


## 5. 结果解读：旧 ID 成功查询不等于动作语义正确

基线六次查询都能找到 `n1`，但六次都点错；安全方案六次都在新 snapshot 中找到目标并通过前置条件。逐案例对照保留错误按钮与正确按钮名称，让 stale action 的业务风险可见。

In [5]:
comparison = []  # 构造同页面上的 stale 基线与安全方案对照
for baseline, safe in zip(baseline_rows, safe_rows):  # 对齐每个任务的两种执行结果
    comparison.append({"案例": baseline["案例"], "意图": baseline["意图"], "旧ID点击": baseline["实际点击"], "重定位点击": safe["执行元素"], "基线正确": baseline["命中目标"], "安全方案正确": safe["命中目标"]})  # 保存逐案例业务语义差异
baseline_hits = sum(row["基线正确"] for row in comparison)  # 统计旧 node_id 方案真正命中意图的次数
safe_hits = sum(row["安全方案正确"] for row in comparison)  # 统计重新观察与验证方案的命中次数
print("DOM stale action 逐案例对照：")  # 输出结果解读标题
pprint(comparison, sort_dicts=False)  # 展示查询成功但语义错误与安全重定位的差异
print(f"正确动作数：旧node基线={baseline_hits}/{len(cases)}，重新观察方案={safe_hits}/{len(cases)}")  # 汇总同数据上的动作正确率

DOM stale action 逐案例对照：
[{'案例': 'B01',
  '意图': '提交金额 699 元的订单',
  '旧ID点击': '返回购物车',
  '重定位点击': '提交订单',
  '基线正确': False,
  '安全方案正确': True},
 {'案例': 'B02',
  '意图': '删除标题为周报的草稿',
  '旧ID点击': '删除全部',
  '重定位点击': '删除草稿',
  '基线正确': False,
  '安全方案正确': True},
 {'案例': 'B03',
  '意图': '预订 B201 会议室',
  '旧ID点击': '取消预订',
  '重定位点击': '预订 B201',
  '基线正确': False,
  '安全方案正确': True},
 {'案例': 'B04',
  '意图': '禁用测试用户 U17',
  '旧ID点击': '禁用管理员',
  '重定位点击': '禁用 U17',
  '基线正确': False,
  '安全方案正确': True},
 {'案例': 'B05',
  '意图': '把商品 P3 数量更新为 2',
  '旧ID点击': '清空购物车',
  '重定位点击': '更新数量',
  '基线正确': False,
  '安全方案正确': True},
 {'案例': 'B06',
  '意图': '接受当前隐私条款',
  '旧ID点击': '拒绝全部',
  '重定位点击': '接受条款',
  '基线正确': False,
  '安全方案正确': True}]
正确动作数：旧node基线=0/6，重新观察方案=6/6


## 6. 失败案例与修正：目标仍存在，但金额变化且按钮被禁用

重新定位不是万能的。订单目标 stable_key 仍在，但名称从“提交订单”变成“支付 ¥799”且 enabled=False；如果只按 stable_key 点击仍会违背用户授权。验证函数会显式停止，修正路径是重新规划并向用户确认新金额，而不是寻找名称相似的按钮。

In [6]:
changed_nodes = [dict(node) for node in cases[0]["after"]]  # 深复制订单页面节点以注入执行前业务变化
changed_target = next(node for node in changed_nodes if node["stable_key"] == "checkout-submit")  # 找到最新页面中的订单目标
changed_target["name"] = "支付 ¥799"  # 模拟价格变化导致按钮可访问名称更新
changed_target["enabled"] = False  # 模拟页面要求重新确认后暂时禁用支付按钮
failure_message = ""  # 初始化安全状态机的失败原因
try:  # 尝试执行基于旧金额与旧名称的动作计划
    resolve_and_verify(changed_nodes, cases[0]["plan"])  # 在业务条件已变化的 snapshot 上重新定位并校验
except ValueError as error:  # 捕获前置条件漂移而不是继续点击
    failure_message = str(error)  # 保存错误详情供 Agent 重规划与审计
repair_action = "停止点击，刷新订单摘要，并请求用户确认 ¥799"  # 定义不会越过授权边界的修正动作
print({"失败复现": failure_message, "最新目标": changed_target, "修正动作": repair_action, "实际点击次数": 0})  # 展示停止执行而非误点的安全结果

{'失败复现': "precondition_changed:{'role': 'button', 'name': '支付 ¥799', 'enabled': False}", '最新目标': {'node_id': 'n9', 'stable_key': 'checkout-submit', 'role': 'button', 'name': '支付 ¥799', 'enabled': False}, '修正动作': '停止点击，刷新订单摘要，并请求用户确认 ¥799', '实际点击次数': 0}


## 7. 生产差距与最小回归检查

真实浏览器需要使用 CDP backend node、Playwright locator 或可访问树，并处理 iframe、shadow root、虚拟列表和导航后的新 execution context。验证与 click 之间仍有 TOCTOU 竞态，高风险动作应尽可能用浏览器原子 locator、页面侧条件检查或二次确认。动作 receipt 还应保存脱敏截图、URL、frame、用户授权和网络响应。最后的断言只守住本实验已展示的 stale ID、snapshot 变化、语义重定位和条件漂移停止。

In [7]:
assert len(cases) >= 5  # 确认真实浏览器动作数量满足逐样本教学要求
assert baseline_hits == 0  # 确认旧 node_id 即使存在也全部指向错误业务元素
assert safe_hits == len(cases)  # 确认重新观察与严格验证后全部命中真实意图
assert first_before_hash != first_after_hash  # 确认订单页面在计划与执行之间确实发生变化
assert all(row["snapshot"] == receipts[row["案例"]]["snapshot"] for row in safe_rows)  # 确认每个动作都绑定最新 snapshot 回执
assert failure_message.startswith("precondition_changed")  # 确认金额或 enabled 漂移会阻止动作执行
assert repair_action.startswith("停止点击")  # 确认修正路径不会静默寻找相似按钮
print("回归检查通过：DOM 版本、语义重定位、动作回执与高风险停止条件均已验证。")  # 输出最终验收结论

回归检查通过：DOM 版本、语义重定位、动作回执与高风险停止条件均已验证。
